In [11]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split


In [12]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = "bilstm_tokenizer"
X_PATH = f"{DATA_DIR}/X.npy"
Y_PATH = f"{DATA_DIR}/y.npy"

MODEL_OUT = "model_warmstart_bilstm2.pt"

# Training hyperparams (RAM-safe)
BATCH_SIZE = 32
EPOCHS = 75
LR = 1e-3

# Model hyperparams
EMB_DIM = 128
HIDDEN_DIM = 128
DROPOUT = 0.2


In [13]:
X = np.load(X_PATH)  # int sequences
y = np.load(Y_PATH).astype(np.float32)

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)

num_labels = y.shape[1]
vocab_size = int(X.max()) + 1  # IMPORTANT: berdasarkan max token id
print("vocab_size:", vocab_size, "num_labels:", num_labels)


X: (12882, 100) int32
y: (12882, 12) float32
vocab_size: 20000 num_labels: 12


In [14]:
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.1, random_state=42
)

print("Train:", X_tr.shape, y_tr.shape)
print("Val  :", X_va.shape, y_va.shape)


Train: (11593, 100) (11593, 12)
Val  : (1289, 100) (1289, 12)


In [15]:
class NumpyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = NumpyDataset(X_tr, y_tr)
val_ds   = NumpyDataset(X_va, y_va)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


In [16]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_labels, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)  # *2 karena bidirectional

    def forward(self, x):
        # x: (B, L)
        emb = self.embedding(x)              # (B, L, E)
        out, _ = self.lstm(emb)              # (B, L, 2H)
        last = out[:, -1, :]                 # (B, 2H) ambil timestep terakhir (simple & aman)
        last = self.dropout(last)
        logits = self.fc(last)               # (B, num_labels)
        return logits


In [17]:
model = BiLSTMClassifier(
    vocab_size=vocab_size,
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    num_labels=num_labels,
    dropout=DROPOUT
).to(DEVICE)

model


BiLSTMClassifier(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=256, out_features=12, bias=True)
)

In [18]:
# pos_weight = (#neg / #pos) per label
pos = y_tr.sum(axis=0)                 # (num_labels,)
neg = y_tr.shape[0] - pos
pos_weight = (neg / (pos + 1e-8)).astype(np.float32)

pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)


In [19]:
def run_eval(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            bs = xb.size(0)
            total_loss += loss.item() * bs
            n += bs
    return total_loss / max(n, 1)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    n = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        n += bs

    train_loss = total_loss / max(n, 1)
    val_loss = run_eval(model, val_loader, criterion)

    print(f"Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")


Epoch 1/75 | train_loss=1.1462 | val_loss=1.1468
Epoch 2/75 | train_loss=1.1459 | val_loss=1.1470
Epoch 3/75 | train_loss=1.1457 | val_loss=1.1473
Epoch 4/75 | train_loss=1.1459 | val_loss=1.1475
Epoch 5/75 | train_loss=1.1458 | val_loss=1.1476
Epoch 6/75 | train_loss=1.1457 | val_loss=1.1478
Epoch 7/75 | train_loss=1.1457 | val_loss=1.1475
Epoch 8/75 | train_loss=1.1457 | val_loss=1.1477
Epoch 9/75 | train_loss=1.1457 | val_loss=1.1478
Epoch 10/75 | train_loss=1.1457 | val_loss=1.1471
Epoch 11/75 | train_loss=1.1456 | val_loss=1.1478
Epoch 12/75 | train_loss=1.1457 | val_loss=1.1477
Epoch 13/75 | train_loss=1.1456 | val_loss=1.1477
Epoch 14/75 | train_loss=1.1456 | val_loss=1.1473
Epoch 15/75 | train_loss=1.1457 | val_loss=1.1475
Epoch 16/75 | train_loss=1.1456 | val_loss=1.1475
Epoch 17/75 | train_loss=1.1456 | val_loss=1.1477
Epoch 18/75 | train_loss=1.1456 | val_loss=1.1475
Epoch 19/75 | train_loss=1.1456 | val_loss=1.1475
Epoch 20/75 | train_loss=1.1456 | val_loss=1.1476
Epoch 21/

In [20]:
ckpt = {
    "model_state_dict": model.state_dict(),
    "vocab_size": vocab_size,
    "emb_dim": EMB_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_labels": num_labels,
    "max_len": X.shape[1],
}

torch.save(ckpt, MODEL_OUT)
print("Saved:", MODEL_OUT)


Saved: model_warmstart_bilstm2.pt
